In [2]:
import shutil
shutil.rmtree("/zfsauton2/home/mineuih/.cache/flashinfer", ignore_errors=True)

In [1]:
import os

os.environ["LD_LIBRARY_PATH"] = "/usr/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["LIBRARY_PATH"] = "/usr/lib64:" + os.environ.get("LIBRARY_PATH", "")

In [2]:
from vllm import LLM, SamplingParams
llm = LLM(
    model="google/gemma-4-E2B-it",
    dtype="float16",
    gpu_memory_utilization=0.9,
    tensor_parallel_size=1,
    max_model_len=4096,
)


INFO 05-19 01:28:01 [utils.py:240] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'google/gemma-4-E2B-it'}
INFO 05-19 01:28:01 [model.py:568] Resolved architecture: Gemma4ForConditionalGeneration
WARNING 05-19 01:28:01 [model.py:2035] Casting torch.bfloat16 to torch.float16.
INFO 05-19 01:28:01 [model.py:1697] Using max model len 4096
INFO 05-19 01:28:01 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-19 01:28:01 [config.py:101] Gemma4 model has heterogeneous head dimensions (head_dim=256, global_head_dim=512). Forcing TRITON_ATTN backend to prevent mixed-backend numerical divergence.
INFO 05-19 01:28:01 [vllm.py:886] Asynchronous scheduling is enabled.
INFO 05-19 01:28:01 [kernel.py:212] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=14029) INFO 05-19 01:28:26 [core.py:109

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore pid=14029) INFO 05-19 01:28:33 [weight_utils.py:872] Prefetching checkpoint files: 10% (1/1)
(EngineCore pid=14029) INFO 05-19 01:28:33 [weight_utils.py:895] Prefetching checkpoint files into page cache finished in 1.83s
(EngineCore pid=14029) INFO 05-19 01:28:34 [default_loader.py:397] Loading weights took 2.90 seconds
(EngineCore pid=14029) INFO 05-19 01:28:34 [gpu_model_runner.py:4959] Model loading took 9.9 GiB memory and 3.691156 seconds
(EngineCore pid=14029) INFO 05-19 01:28:35 [gpu_model_runner.py:5920] Encoder cache will be initialized with a budget of 8192 tokens, and profiled with 3 video items of the maximum feature size.
(EngineCore pid=14029) WARNING 05-19 01:28:35 [op.py:290] Priority not set for op rms_norm, using native implementation.
(EngineCore pid=14029) INFO 05-19 01:28:42 [backends.py:1089] Using cache directory: /zfsauton2/home/mineuih/.cache/vllm/torch_compile_cache/3acf5c0858/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=14029) INFO 05

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 23.59it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:05<00:00,  6.11it/s]


(EngineCore pid=14029) INFO 05-19 01:30:12 [gpu_model_runner.py:6243] Graph capturing finished in 9 secs, took 0.61 GiB
(EngineCore pid=14029) INFO 05-19 01:30:12 [gpu_worker.py:621] CUDA graph pool memory: 0.61 GiB (actual), 0.62 GiB (estimated), difference: 0.01 GiB (1.9%).
(EngineCore pid=14029) INFO 05-19 01:30:12 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(EngineCore pid=14029) INFO 05-19 01:30:12 [core.py:299] init engine (profile, create kv cache, warmup model) took 97.50 s (compilation: 6.86 s)
(EngineCore pid=14029) INFO 05-19 01:30:13 [kernel.py:212] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 모델 설정
model_name = "google/gemma-4-E2B-it"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model on {device}...")

# 모델 로드 (최적화됨)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()

# BOS 토큰 설정
tokenizer.add_bos_token = True

print("Model loaded successfully!")
print(f"BOS token: {tokenizer.bos_token_id}, EOS token: {tokenizer.eos_token_id}")

def generate(prompt: str, max_tokens: int = 256, temperature: float = 0.7, top_p: float = 0.9):
    full_prompt = f"<bos>{prompt}" 
    
    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2,  # 반복 제약
            no_repeat_ngram_size=2,  # 2-gram 반복 금지
            length_penalty=1.0
        )

    generated_ids = outputs[0][inputs.input_ids.shape[-1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    
    return generated_text.strip()

def make_prompt(behavior_summary: str) -> str:
    prompt = f"""Convert the following ego vehicle behavior summary into a short driving instruction for a Vision Language Agent (VLA). 
The instruction should be concise and direct, starting with an action verb.
Output only the instruction without any additional explanation or context.
example: "Ego vehicle 0 continues straight forward along its visible future trajectory." -> "Go straight forward."
Behavior summary: {behavior_summary}
Driving instruction:"""
    return prompt

In [1]:
from pathlib import Path
import json
import os
from tqdm import tqdm

file_paths = ["/zfsauton/scratch/eshau/gemma4_31b_output/gemma4_waymax_structured_full_imgs_s%d.jsonl" % i for i in range(9)]
indices = dict()
for i, file_path in enumerate(file_paths):
    with Path(file_path).open('r', encoding='utf-8') as f:
        for idx, line in tqdm(enumerate(f), desc=f'Processing {file_path}'):
            data = json.loads(line)
            png_name = os.path.basename(data['image_path'])
            tfrecord_i = int(png_name.split('_')[1])
            scenario_i = int(png_name.split('_')[3])
            start_i = int(png_name.split('_')[5])
            if tfrecord_i not in indices:
                indices[tfrecord_i] = {scenario_i: {start_i: [(i, idx)]}}
            elif scenario_i not in indices[tfrecord_i]:
                indices[tfrecord_i][scenario_i] = {start_i: [(i, idx)]}
            elif start_i not in indices[tfrecord_i][scenario_i]:
                indices[tfrecord_i][scenario_i][start_i] = [(i, idx)]
            else:
                indices[tfrecord_i][scenario_i][start_i].append((i, idx))

Processing /zfsauton/scratch/eshau/gemma4_31b_output/gemma4_waymax_structured_full_imgs_s0.jsonl: 0it [00:00, ?it/s]

Processing /zfsauton/scratch/eshau/gemma4_31b_output/gemma4_waymax_structured_full_imgs_s0.jsonl: 1071455it [00:15, 70180.08it/s]
Processing /zfsauton/scratch/eshau/gemma4_31b_output/gemma4_waymax_structured_full_imgs_s1.jsonl: 1071455it [00:15, 71282.14it/s]
Processing /zfsauton/scratch/eshau/gemma4_31b_output/gemma4_waymax_structured_full_imgs_s2.jsonl: 1071450it [00:14, 73292.99it/s]
Processing /zfsauton/scratch/eshau/gemma4_31b_output/gemma4_waymax_structured_full_imgs_s3.jsonl: 1071450it [00:15, 69540.01it/s]
Processing /zfsauton/scratch/eshau/gemma4_31b_output/gemma4_waymax_structured_full_imgs_s4.jsonl: 1071450it [00:15, 69630.84it/s]
Processing /zfsauton/scratch/eshau/gemma4_31b_output/gemma4_waymax_structured_full_imgs_s5.jsonl: 1071450it [00:15, 70561.50it/s]
Processing /zfsauton/scratch/eshau/gemma4_31b_output/gemma4_waymax_structured_full_imgs_s6.jsonl: 1071450it [00:14, 74778.81it/s]
Processing /zfsauton/scratch/eshau/gemma4_31b_output/gemma4_waymax_structured_full_imgs_s7

In [6]:
save_dir = "/zfsauton/scratch/mineuih/waymax_rs/instructions/training"
for tfrecord_i in range(1000):
    save_file = os.path.join(save_dir, f'tfrecord_{tfrecord_i}.jsonl')
    num_scenarios = len(indices[tfrecord_i])
    for scenario_i in tqdm(range(num_scenarios)):
        scenario_dict = {"scenario_index": scenario_i}
        start_indices = sorted(indices[tfrecord_i][scenario_i].keys())
        for start_i in start_indices:
            scenario_dict[start_i] = {"summary": []}
            summary_is = indices[tfrecord_i][scenario_i][start_i]
            for summary_i in summary_is:
                file_i, line_i = summary_i
                with Path(file_paths[file_i]).open('r', encoding='utf-8') as f:
                    for idx, line in enumerate(f):
                        if idx == line_i:
                            data = json.loads(line)
                            scenario_dict[start_i]["summary"].append(data["parsed"]["trajectory_summary"])
                            break
        if not os.path.exists(save_file):
            with open(save_file, 'w', encoding='utf-8') as f:
                f.write(json.dumps(scenario_dict) + '\n')
        else:
            with open(save_file, 'a', encoding='utf-8') as f:
                f.write(json.dumps(scenario_dict) + '\n')



 16%|█▌        | 80/514 [03:27<18:44,  2.59s/it]


KeyboardInterrupt: 